# Kaggle FLUX.1 Schnell Image Notebook

Reads `job.json`, generates one high-quality image with FLUX.1 Schnell, and writes `output.png`.

## Install Dependencies

In [ ]:
print("Installing packages...")

import json
import subprocess
import sys

packages = [
    "diffusers==0.35.1",
    "transformers==4.56.2",
    "accelerate==1.11.0",
    "gguf>=0.10.0",
    "safetensors>=0.4.3",
    "ftfy>=6.3.1",
    "invisible-watermark>=0.2.0",
]


def detect_cuda_capability():
    probe = "import json, torch; print(json.dumps(torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None))"
    result = subprocess.run(
        [sys.executable, "-c", probe],
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return None
    return json.loads(result.stdout.strip())


cuda_capability = detect_cuda_capability()
if cuda_capability:
    print(f"Detected CUDA capability: {cuda_capability[0]}.{cuda_capability[1]}")

# Some Kaggle images ship a newer PyTorch build that no longer includes P100 (sm_60) kernels.
# If Kaggle assigns a Pascal GPU, install a CUDA 12.4 wheel that still supports it before importing torch.
if cuda_capability and int(cuda_capability[0]) < 7:
    print("Older CUDA GPU detected; installing a PyTorch build with broader GPU architecture support...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--force-reinstall",
        "--no-deps",
        "torch==2.5.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
    ])

# FLUX generation does not require torchvision/torchaudio. Removing them avoids Kaggle package mismatches.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "--quiet", "torchvision", "torchaudio"],
    check=False,
)

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages])

print("Dependencies installed.")

## Import Libraries

In [ ]:
from __future__ import annotations

import gc
import json
import os
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from diffusers import FluxPipeline, FluxTransformer2DModel, GGUFQuantizationConfig
from PIL import Image

print("Libraries imported.")

## Read Configuration

In [ ]:
MODEL_ID = "black-forest-labs/FLUX.1-schnell"
GGUF_MODEL_ID = "city96/FLUX.1-schnell-gguf"
OUTPUT_PATH = Path("output.png")
GGUF_FILES = {
    "Q2_K": "flux1-schnell-Q2_K.gguf",
    "Q3_K_S": "flux1-schnell-Q3_K_S.gguf",
    "Q4_K_S": "flux1-schnell-Q4_K_S.gguf",
}


@dataclass(frozen=True)
class ImageJobConfig:
    prompt: str
    negative_prompt: str
    width: int
    height: int
    num_inference_steps: int
    guidance_scale: float
    max_sequence_length: int
    hf_token: str | None
    gguf_quant: str


def find_job_file(filename: str = "job.json") -> Path:
    current_dir_path = Path(filename)
    if current_dir_path.exists():
        return current_dir_path

    kaggle_input_dir = Path("/kaggle/input")
    if kaggle_input_dir.exists():
        matches = sorted(kaggle_input_dir.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError("Could not find job.json in the current directory or /kaggle/input.")


def get_cuda_major() -> int | None:
    if not torch.cuda.is_available():
        return None
    major, _minor = torch.cuda.get_device_capability(0)
    return int(major)


def default_dimension() -> int:
    cuda_major = get_cuda_major()
    if cuda_major is not None and cuda_major < 7:
        return 768
    return 1024


def default_gguf_quant() -> str:
    cuda_major = get_cuda_major()
    if cuda_major is not None and cuda_major < 7:
        return "Q3_K_S"
    return "Q4_K_S"


def clamp_dimension(value: Any, default: int) -> int:
    try:
        dimension = int(value)
    except (TypeError, ValueError):
        dimension = default

    max_dimension = 768 if get_cuda_major() is not None and get_cuda_major() < 7 else 1344
    dimension = max(512, min(max_dimension, dimension))
    return max(512, (dimension // 16) * 16)


def clamp_int(value: Any, default: int, minimum: int, maximum: int) -> int:
    try:
        parsed = int(value)
    except (TypeError, ValueError):
        parsed = default
    return max(minimum, min(maximum, parsed))


def clamp_float(value: Any, default: float, minimum: float, maximum: float) -> float:
    try:
        parsed = float(value)
    except (TypeError, ValueError):
        parsed = default
    return max(minimum, min(maximum, parsed))


def normalize_gguf_quant(value: Any) -> str:
    quant = str(value or default_gguf_quant()).strip().upper()
    if quant not in GGUF_FILES:
        print(f"Unsupported gguf_quant={quant!r}; using {default_gguf_quant()}.")
        return default_gguf_quant()
    return quant


def read_config(path: Path) -> ImageJobConfig:
    raw = json.loads(path.read_text(encoding="utf-8"))
    prompt = str(raw.get("prompt", "")).strip()
    if not prompt:
        raise ValueError("job.json must contain a non-empty 'prompt' field.")

    return ImageJobConfig(
        prompt=prompt,
        negative_prompt=str(
            raw.get(
                "negative_prompt",
                "blurry, low quality, distorted, watermark, text artifacts",
            )
        ).strip(),
        width=clamp_dimension(raw.get("width"), default_dimension()),
        height=clamp_dimension(raw.get("height"), default_dimension()),
        num_inference_steps=clamp_int(raw.get("num_inference_steps"), 4, 1, 8),
        guidance_scale=clamp_float(raw.get("guidance_scale"), 0.0, 0.0, 0.0),
        max_sequence_length=clamp_int(raw.get("max_sequence_length"), 256, 64, 256),
        hf_token=read_hf_token(raw),
        gguf_quant=normalize_gguf_quant(raw.get("gguf_quant")),
    )


def redact_job_token(path: Path) -> None:
    try:
        raw = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return

    redacted = False
    for key in ("hf_token", "HF_TOKEN", "huggingface_token"):
        if key in raw:
            raw[key] = "[redacted]"
            redacted = True

    if redacted:
        try:
            path.write_text(json.dumps(raw, indent=2), encoding="utf-8")
            print("Redacted token fields from local job.json.")
        except Exception as error:
            print(f"Could not redact local job.json: {error}")


HF_TOKEN_NAMES = (
    "HF_TOKEN",
    "hf_token",
    "HUGGING_FACE_HUB_TOKEN",
    "HUGGINGFACE_HUB_TOKEN",
    "HUGGINGFACE_TOKEN",
    "KAGGLE_HF_TOKEN",
)


def read_hf_token(raw_config: dict[str, Any]) -> str | None:
    for config_name in ("hf_token", "HF_TOKEN", "huggingface_token"):
        config_token = str(raw_config.get(config_name, "")).strip()
        if config_token:
            print(f"Using Hugging Face token from job.json field '{config_name}'.")
            return config_token

    for env_name in HF_TOKEN_NAMES:
        env_token = os.environ.get(env_name, "").strip()
        if env_token:
            print(f"Using Hugging Face token from environment variable '{env_name}'.")
            return env_token

    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        for secret_name in HF_TOKEN_NAMES:
            try:
                secret_token = str(secrets.get_secret(secret_name) or "").strip()
            except Exception:
                secret_token = ""
            if secret_token:
                print(f"Using Hugging Face token from Kaggle Secret '{secret_name}'.")
                return secret_token
    except Exception as error:
        print(f"Kaggle Secrets are unavailable in this environment: {error}")

    return None


print("Configuration helpers ready.")

## Load Model

In [ ]:
def get_model_dtype() -> torch.dtype:
    if not torch.cuda.is_available():
        return torch.float32

    major, _minor = torch.cuda.get_device_capability(0)
    if major >= 8:
        return torch.bfloat16
    return torch.float16


def optimize_pipeline(pipeline: Any) -> Any:
    if hasattr(pipeline, "enable_model_cpu_offload"):
        pipeline.enable_model_cpu_offload()
    elif hasattr(pipeline, "enable_sequential_cpu_offload"):
        pipeline.enable_sequential_cpu_offload()
    elif torch.cuda.is_available():
        pipeline.to("cuda")

    if hasattr(pipeline, "enable_attention_slicing"):
        pipeline.enable_attention_slicing()
    if hasattr(pipeline, "enable_vae_slicing"):
        pipeline.enable_vae_slicing()
    if hasattr(pipeline, "enable_vae_tiling"):
        pipeline.enable_vae_tiling()

    return pipeline


def build_gguf_url(gguf_quant: str) -> str:
    filename = GGUF_FILES[gguf_quant]
    return f"https://huggingface.co/{GGUF_MODEL_ID}/blob/main/{filename}"


def load_flux_pipeline(config: ImageJobConfig) -> FluxPipeline:
    model_dtype = get_model_dtype()
    print(f"Loading FLUX.1 Schnell model with dtype={model_dtype}...")
    if config.hf_token is None:
        raise RuntimeError(
            "FLUX.1 Schnell is gated on Hugging Face. Accept the model terms on Hugging Face "
            "and provide a token via Kaggle Secret HF_TOKEN, environment variable HF_TOKEN, "
            "or job.json field hf_token."
        )
    gguf_url = build_gguf_url(config.gguf_quant)
    print(f"Loading FLUX GGUF transformer ({config.gguf_quant})...")
    quantization_config = GGUFQuantizationConfig(compute_dtype=model_dtype)
    transformer = FluxTransformer2DModel.from_single_file(
        gguf_url,
        config=MODEL_ID,
        subfolder="transformer",
        quantization_config=quantization_config,
        torch_dtype=model_dtype,
        token=config.hf_token,
    )

    print("Loading remaining FLUX pipeline components...")
    pipeline = FluxPipeline.from_pretrained(
        MODEL_ID,
        transformer=transformer,
        torch_dtype=model_dtype,
        use_safetensors=True,
        token=config.hf_token,
    )
    return optimize_pipeline(pipeline)


def release_model(model: Any | None) -> None:
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Model helpers ready.")

## Generate Image

In [ ]:
def generate_image(config: ImageJobConfig) -> Image.Image:
    pipeline = load_flux_pipeline(config)
    print("Generating image with FLUX.1 Schnell...")
    if config.negative_prompt:
        print("Note: FLUX.1 Schnell does not use negative_prompt in the standard Diffusers pipeline; ignoring it.")

    try:
        with torch.inference_mode():
            image = pipeline(
                prompt=config.prompt,
                width=config.width,
                height=config.height,
                num_inference_steps=config.num_inference_steps,
                guidance_scale=config.guidance_scale,
                max_sequence_length=config.max_sequence_length,
                output_type="pil",
            ).images[0]
        return image.convert("RGB")
    finally:
        release_model(pipeline)


print("Generation helpers ready.")

## Save Output

In [ ]:
def save_image(image: Image.Image, path: Path) -> None:
    print("Saving output.png...")
    image.save(path, format="PNG", optimize=True)
    print(f"Saved {path} ({path.stat().st_size} bytes).")


print("Save helper ready.")

## Upload Placeholder

In [ ]:
def upload_image(path: Path) -> str:
    return "https://example.invalid/output.png"


print("Upload placeholder ready.")

## Cleanup

In [ ]:
def cleanup() -> None:
    print("Cleaning up...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Cleanup complete.")


print("Cleanup helper ready.")

## Run Pipeline

In [ ]:
def main() -> None:
    final_image: Image.Image | None = None

    try:
        print("Reading prompt...")
        job_path = find_job_file("job.json")
        config = read_config(job_path)
        redact_job_token(job_path)
        print(f"Prompt loaded from {job_path}.")
        print(f"Image size: {config.width}x{config.height}")
        print(f"GGUF quantization: {config.gguf_quant}")

        final_image = generate_image(config)

        save_image(final_image, OUTPUT_PATH)
        upload_url = upload_image(OUTPUT_PATH)
        print(f"Upload placeholder URL: {upload_url}")
        print("Finished.")
    except Exception as error:
        print("High-quality image job failed.")
        print(f"Error: {error}")
        traceback.print_exc(limit=2)
        raise RuntimeError(str(error)) from None
    finally:
        del final_image
        cleanup()


main()